# STEP 2 — UTCI daily maximum °C + mask Jawa (1996–2025)
Proses streaming **satu file per hari**: 24 jam UTC → maksimum → konversi Kelvin ke °C → crop + mask **Jawa** → NetCDF. Tidak memuat ribuan file sekaligus.

**Prioritas mask:** `D:\Daps\Data\Hasil_Tmax\00_Jawa_mask_native_grid.nc` jika tersedia; jika tidak, gunakan gabungan poligon provinsi Jawa (Natural Earth admin-1). Mask hasilnya juga diekspor untuk dipakai pada Step 4–5. Mask dari Tmax sangat dianjurkan agar Tmax dan UTCI konsisten.

**Keamanan file:** menulis ke file `.part.nc` terlebih dahulu, memeriksa hasilnya, kemudian mengubah nama menjadi output final. Tidak menghapus file hasil lama di awal (menghindari kerusakan saat terjadi error/PermissionError).

In [1]:
from pathlib import Path
import os, gc
import numpy as np
import pandas as pd
import xarray as xr
from netCDF4 import Dataset

BASE = Path(r"D:\Daps\Data")
RAW_DIR = BASE / "UTCI" / "1996-2025"
OUT_DIR = BASE / "UTCI" / "Hasil_UTCI_Jawa_1996_2025"
TMAX_MASK_CANDIDATES = [BASE / "Hasil_Tmax" / n for n in (
    '00_Jawa_mask_native_grid.nc', '00_jawa_mask_native_grid.nc', '00_java_mask_native_grid.nc')]
# Jika punya shapefile Natural Earth 10m lokal, isi PATH ini (opsional).
SHAPEFILE_ADMIN1 = None  # contoh: Path(r'D:\Daps\Data\Batas\ne_10m_admin_1_states_provinces.shp')
START_YEAR, END_YEAR = 1996, 2025
INPUT_UNITS = 'auto'    # 'auto', 'kelvin', atau 'celsius'
FILL = np.float32(-9999)
QC1 = OUT_DIR / '01_qc_utci_raw_files_1996_2025.csv'
QC1_STATUS = OUT_DIR / '02_qc_utci_step1_status.csv'
OUTPUT = OUT_DIR / '03_utci_dailymax_1996_2025_C_Jawa.nc'
QC2 = OUT_DIR / '04_qc_utci_dailymax_1996_2025.csv'
MASK_OUT = OUT_DIR / '00_Jawa_mask_native_grid.nc'
OUT_DIR.mkdir(parents=True, exist_ok=True)
if not RAW_DIR.is_dir(): raise FileNotFoundError(RAW_DIR)
if not QC1.exists() or not QC1_STATUS.exists():
    raise FileNotFoundError('Jalankan STEP 1 terlebih dahulu; CSV QC/status belum ada.')
status = pd.read_csv(QC1_STATUS)
if not bool(status.passed.all()): raise RuntimeError('STEP 1 belum LULUS.')
print('Input:', RAW_DIR, '\nOutput:', OUTPUT)

Input: D:\Daps\Data\UTCI\1996-2025 
Output: D:\Daps\Data\UTCI\Hasil_UTCI_Jawa_1996_2025\03_utci_dailymax_1996_2025_C_Jawa.nc


## 1. File kronologis dari QC STEP 1 dan standardisasi koordinat

In [2]:
qc1 = pd.read_csv(QC1, parse_dates=['date'])
qc1 = qc1.sort_values('date').reset_index(drop=True)
files = [Path(p) for p in qc1.path]
expected = pd.date_range(f'{START_YEAR}-01-01', f'{END_YEAR}-12-31', freq='D')
if len(files) != len(expected) or not pd.DatetimeIndex(qc1.date).equals(expected):
    raise ValueError('Tanggal CSV QC STEP 1 tidak cocok dengan periode 1996–2025.')
missing = [str(p) for p in files if not p.exists()]
if missing: raise FileNotFoundError(f'File raw tak ditemukan: {missing[:5]}')

def standardize(da):
    aliases = {'latitude':('latitude','lat'), 'longitude':('longitude','lon'), 'time':('time','valid_time')}
    rename = {}
    for dst, choices in aliases.items():
        found = next((k for k in choices if k in da.dims), None)
        if found is None: raise ValueError(f'Dimensi {dst} tidak ada: {da.dims}')
        if found != dst: rename[found]=dst
    return da.rename(rename).transpose('time','latitude','longitude')

with xr.open_dataset(files[0], decode_timedelta=False) as ds:
    var = 'utci' if 'utci' in ds else 'UTCI' if 'UTCI' in ds else None
    if var is None: raise KeyError('Variabel utci tidak tersedia')
    da = standardize(ds[var]); full_lat = da.latitude.values.astype(float); full_lon = da.longitude.values.astype(float)
    sample_mean = float(da.mean(skipna=True).values)
    unit_attr = str(da.attrs.get('units','')).strip().lower()
print('Grid sumber:', (len(full_lat),len(full_lon)), 'mean raw sampel:', sample_mean,'unit:',unit_attr)

Grid sumber: (15, 45) mean raw sampel: 299.337890625 unit: 


## 2. Buat mask yang cocok dengan grid sumber (Tmax Jawa → cadangan poligon Jawa)

In [3]:
mask_path = next((p for p in TMAX_MASK_CANDIDATES if p.exists()), None)
if mask_path is not None:
    with xr.open_dataset(mask_path, decode_timedelta=False) as mds:
        mvar = next((k for k in ('java_mask','jawa_mask','Jawa_mask','mask','land_mask') if k in mds.data_vars), None)
        if mvar is None:
            candidates = [k for k in mds.data_vars if mds[k].ndim == 2]
            if len(candidates)!=1: raise ValueError(f'Variabel mask ambigu: {list(mds.data_vars)}')
            mvar = candidates[0]
        mm = mds[mvar]
        aliases={}
        if 'lat' in mm.dims: aliases['lat']='latitude'
        if 'lon' in mm.dims: aliases['lon']='longitude'
        mm = mm.rename(aliases).transpose('latitude','longitude')
        mask_lat=mm.latitude.values.astype(float);mask_lon=mm.longitude.values.astype(float)
        mask_grid=np.asarray(mm.fillna(0).values != 0, dtype=bool)
    print('Sumber mask: Tmax Jawa →',mask_path)
else:
    try:
        import cartopy.io.shapereader as shpreader
        from shapely.ops import unary_union
        from shapely import contains_xy
    except ImportError as exc:
        raise ImportError('Tidak ada mask Tmax Jawa. Instal cartopy dan shapely untuk membangun mask poligon.') from exc
    shp = Path(SHAPEFILE_ADMIN1) if SHAPEFILE_ADMIN1 else shpreader.natural_earth(
        resolution='10m', category='cultural',name='admin_1_states_provinces')
    keys=('jawa','java','jakarta','djakarta','yogyakarta','banten')
    geoms=[]
    for record in shpreader.Reader(shp).records():
        attrs=record.attributes
        country=' '.join(str(attrs.get(k,'')).lower() for k in ('admin','geonunit','name_en','name'))
        if 'indonesia' not in country: continue
        names=' '.join(str(attrs.get(k,'')).lower() for k in ('name','name_en','name_local','gn_name'))
        if any(k in names for k in keys): geoms.append(record.geometry)
    if not geoms: raise ValueError('Poligon provinsi Jawa tidak ditemukan di Natural Earth admin-1.')
    geometry = unary_union(geoms)
    bb=(104.5,116.5,-9.5,-5.0)
    ilat=np.where((full_lat>=bb[2]) & (full_lat<=bb[3]))[0]
    ilon=np.where((full_lon>=bb[0]) & (full_lon<=bb[1]))[0]
    mask_lat=full_lat[ilat];mask_lon=full_lon[ilon]
    xx,yy=np.meshgrid(mask_lon,mask_lat)
    mask_grid=np.asarray(contains_xy(geometry,xx,yy),dtype=bool)
    print('Sumber mask: Natural Earth 10m (gabungan provinsi Jawa).')

def exact_indices(source, target, tol=1e-5):
    ix=[]
    for x in target:
        found=np.flatnonzero(np.isclose(source,x,rtol=0,atol=tol))
        if len(found)!=1: raise ValueError(f'Koordinat {x} tidak cocok unik ke grid raw; mask Tmax mungkin memakai resolusi berbeda.')
        ix.append(int(found[0]))
    return np.asarray(ix,dtype=int)

i_lat=exact_indices(full_lat, mask_lat);i_lon=exact_indices(full_lon, mask_lon)
if mask_grid.shape!=(len(mask_lat),len(mask_lon)) or mask_grid.sum()<3:
    raise ValueError('Mask Jawa kosong atau bentuk grid tidak valid.')
print('Mask Jawa:', mask_grid.shape, '| grid valid:',int(mask_grid.sum()))
# simpan 0/1 (bukan bool _FillValue), agar plot tetap konsisten di tahap lain
mask_ds=xr.Dataset({'java_mask':(('latitude','longitude'),mask_grid.astype('uint8'))},
                   coords={'latitude':mask_lat.astype('float32'),'longitude':mask_lon.astype('float32')},
                   attrs={'title':'Java native UTCI grid land mask','source':str(mask_path) if mask_path else 'Natural Earth admin-1'})
mask_tmp=MASK_OUT.with_name(MASK_OUT.stem+'.part.nc')
mask_ds.to_netcdf(mask_tmp, engine='netcdf4')
os.replace(mask_tmp,MASK_OUT)
print('Mask tersimpan:',MASK_OUT)

Sumber mask: Tmax Jawa → D:\Daps\Data\Hasil_Tmax\00_Jawa_mask_native_grid.nc
Mask Jawa: (12, 36) | grid valid: 176
Mask tersimpan: D:\Daps\Data\UTCI\Hasil_UTCI_Jawa_1996_2025\00_Jawa_mask_native_grid.nc


## 3. Konversi unit aman dan uji satu hari

In [4]:
def infer_kelvin(units, sample):
    setting=INPUT_UNITS.strip().lower()
    if setting=='kelvin':return True
    if setting=='celsius':return False
    if setting!='auto':raise ValueError('INPUT_UNITS harus auto, kelvin, atau celsius')
    units=str(units).strip().lower()
    if units in ('k','kelvin','degrees_kelvin','degree_kelvin') or 'kelvin' in units:return True
    if units in ('degc','c','celsius','degree_celsius','degrees_celsius') or 'celsius' in units:return False
    if not np.isfinite(sample):raise ValueError('Sample UTCI raw tidak valid; tentukan INPUT_UNITS manual.')
    return sample>100

IS_KELVIN=infer_kelvin(unit_attr, sample_mean)
with xr.open_dataset(files[0], decode_timedelta=False) as ds:
    da=standardize(ds[var]);d=da.isel(latitude=i_lat,longitude=i_lon)
    daily=np.asarray(d.max('time',skipna=True).values,dtype='float32')
    if IS_KELVIN:daily-=np.float32(273.15)
    daily[~mask_grid]=np.nan
    if not np.isfinite(daily).any():raise ValueError('Sampel daily max seluruhnya NaN')
print('Unit input:', 'Kelvin' if IS_KELVIN else '°C')
print('UTCImax contoh [°C]:',float(np.nanmin(daily)), 's.d.',float(np.nanmax(daily)))

Unit input: Kelvin
UTCImax contoh [°C]: 28.747406005859375 s.d. 38.158294677734375


## 4. Streaming 10.958 hari ke file sementara; validasi lalu commit atomik

In [5]:
tmp=OUTPUT.with_name(OUTPUT.stem+'.part.nc')
if tmp.exists():
    try: tmp.unlink()
    except PermissionError as exc: raise PermissionError(f'Tutup semua notebook yang membuka file sementara: {tmp}') from exc
records=[]
try:
    with Dataset(tmp,'w',format='NETCDF4') as dest:
        dest.createDimension('time',len(files)); dest.createDimension('latitude',len(mask_lat)); dest.createDimension('longitude',len(mask_lon))
        tv=dest.createVariable('time','f8',('time',));tv.units='days since 1970-01-01 00:00:00';tv.calendar='proleptic_gregorian'
        latv=dest.createVariable('latitude','f4',('latitude',));latv[:]=mask_lat;latv.units='degrees_north'
        lonv=dest.createVariable('longitude','f4',('longitude',));lonv[:]=mask_lon;lonv.units='degrees_east'
        uv=dest.createVariable('utci_max','f4',('time','latitude','longitude'),zlib=True,complevel=3,shuffle=True,
                               fill_value=FILL,chunksizes=(min(31,len(files)),len(mask_lat),len(mask_lon)))
        uv.units='degC';uv.long_name='Daily maximum UTCI';uv.daily_reduction='maximum of 24 UTC hourly values'
        dest.title='Daily Maximum UTCI over Jawa 1996–2025';dest.temporal_basis='00:00–23:00 UTC'
        dest.mask_source=str(mask_path) if mask_path else 'Natural Earth 10m admin-1'
        for i,path in enumerate(files):
            with xr.open_dataset(path,decode_times=True,decode_timedelta=False) as ds:
                da=standardize(ds[var]); times=pd.DatetimeIndex(da.time.values)
                if len(times)!=24 or not (times.normalize() == times[0].normalize()).all():
                    raise ValueError(f'{path.name}: bukan satu tanggal dengan 24 jam.')
                if not (np.allclose(da.latitude.values,full_lat,atol=1e-6) and np.allclose(da.longitude.values,full_lon,atol=1e-6)):
                    raise ValueError(f'{path.name}: grid berubah.')
                date=times[0].normalize()
                if date != expected[i]: raise ValueError(f'{path.name}: {date} tidak sama dengan {expected[i]}')
                units_now=str(da.attrs.get('units','')).strip().lower()
                if units_now and infer_kelvin(units_now,sample_mean) != IS_KELVIN:
                    raise ValueError(f'{path.name}: unit tiba-tiba berubah.')
                z=np.asarray(da.isel(latitude=i_lat,longitude=i_lon).max('time',skipna=True).values,dtype='float32')
                if IS_KELVIN:z-=np.float32(273.15)
                land=z[mask_grid];finite=land[np.isfinite(land)]
                if finite.size==0:raise ValueError(f'{path.name}: tidak ada nilai UTCI di mask Jawa.')
                lo,hi=float(finite.min()),float(finite.max())
                if lo < -100 or hi > 100:raise ValueError(f'{path.name}: nilai °C ekstrem/tidak wajar ({lo:.1f}, {hi:.1f}), periksa unit.')
                z[~mask_grid]=FILL
                # NaN interior tetap ditulis sebagai FillValue
                z[~np.isfinite(z)]=FILL
            tv[i]=float((date-pd.Timestamp('1970-01-01'))/pd.Timedelta(days=1))
            uv[i,:,:]=z
            records.append(dict(date=date,file=path.name,utci_min_C=lo,utci_max_C=hi,valid_land_grid=int(finite.size),mask_grid=int(mask_grid.sum())))
            if i<3 or (i+1)%500==0 or i==len(files)-1:
                print(f'[{i+1:05d}/{len(files):05d}] {date.date()} | {lo:.2f}–{hi:.2f} °C')
            if (i+1)%250==0:gc.collect()
    # Validasi sebelum mengganti output lama
    with xr.open_dataset(tmp,decode_timedelta=False) as check:
        out_dates=pd.DatetimeIndex(check.time.values)
        if not out_dates.equals(expected):raise RuntimeError('QC waktu output gagal.')
        if check.utci_max.shape!=(len(expected),len(mask_lat),len(mask_lon)):
            raise RuntimeError('QC bentuk data output gagal.')
    # os.replace bisa PermissionError di Windows bila file output lama masih dibuka.
    try:os.replace(tmp,OUTPUT)
    except PermissionError as exc:
        raise PermissionError(f'File output sedang dikunci: {OUTPUT}. Tutup ds/dataset dari notebook lain, lalu jalankan ulang sel ini. File lama tidak dihapus.') from exc
except Exception:
    print('Proses belum berhasil; jika ada file .part.nc, itu bukan output final.')
    raise
pd.DataFrame(records).to_csv(QC2,index=False)
print('✅ STEP 2 LULUS:',OUTPUT,'\nQC:',QC2)

[00001/10958] 1996-01-01 | 28.75–38.16 °C
[00002/10958] 1996-01-02 | 24.12–36.73 °C
[00003/10958] 1996-01-03 | 25.47–35.42 °C
[00500/10958] 1997-05-14 | 30.02–38.67 °C
[01000/10958] 1998-09-26 | 24.61–37.96 °C
[01500/10958] 2000-02-08 | 22.17–35.26 °C
[02000/10958] 2001-06-22 | 28.50–38.16 °C
[02500/10958] 2002-11-04 | 28.92–41.26 °C
[03000/10958] 2004-03-18 | 27.19–37.82 °C
[03500/10958] 2005-07-31 | 24.46–38.34 °C
[04000/10958] 2006-12-13 | 30.97–40.87 °C
[04500/10958] 2008-04-26 | 28.97–39.41 °C
[05000/10958] 2009-09-08 | 25.85–40.43 °C
[05500/10958] 2011-01-21 | 25.61–38.26 °C
[06000/10958] 2012-06-04 | 26.29–37.78 °C
[06500/10958] 2013-10-17 | 29.36–40.84 °C
[07000/10958] 2015-03-01 | 29.58–38.28 °C
[07500/10958] 2016-07-13 | 30.83–38.21 °C
[08000/10958] 2017-11-25 | 28.11–38.95 °C
[08500/10958] 2019-04-09 | 30.92–40.23 °C
[09000/10958] 2020-08-21 | 27.05–39.86 °C
[09500/10958] 2022-01-03 | 29.70–39.54 °C
[10000/10958] 2023-05-18 | 28.79–37.89 °C
[10500/10958] 2024-09-29 | 28.88–4